# XGBoost — 2M Registros (Produção)

Este notebook treina o modelo final com **todos os 2M de registros** do Lending Club (sem amostragem), visando máxima generalização para produção.

## Melhorias em relação à versão anterior:
- `scale_pos_weight` calculado dinamicamente (não fixado em 4)
- Avaliação no conjunto de **validação** (holdout final)
- **Ajuste de threshold** via `predict_proba` para otimizar F1
- **Feature importance** visualizada
- **Curva ROC / AUC** para avaliação robusta

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    recall_score, roc_auc_score, roc_curve, confusion_matrix
)

print("Bibliotecas carregadas ✔")

In [ ]:
# Carregar 2M COMPLETOS (sem .sample())
dados = pd.read_csv(
    '/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz',
    compression='gzip',
    low_memory=False,
    usecols=[
        'loan_amnt', 'term', 'int_rate', 'installment',
        'annual_inc', 'dti', 'fico_range_high', 'revol_util',
        'delinq_2yrs', 'open_acc', 'pub_rec', 'mort_acc',
        'loan_status'
    ]
)
print(f"Shape completo: {dados.shape}")

In [ ]:
# Filtro e target
dados = dados[dados['loan_status'].isin(['Fully Paid', 'Charged Off'])]
dados['inadimplente'] = (dados['loan_status'] == 'Charged Off').astype(int)

# Renomear
rename_dict = {
    'loan_amnt': 'valor_emprestimo', 'term': 'prazo',
    'int_rate': 'taxa_juros', 'installment': 'parcela',
    'annual_inc': 'renda_anual', 'dti': 'indice_endividamento',
    'fico_range_high': 'score_credito', 'revol_util': 'uso_credito_rotativo',
    'delinq_2yrs': 'inadimplencias_2anos', 'open_acc': 'contas_abertas',
    'pub_rec': 'registros_negativos', 'mort_acc': 'contas_hipoteca'
}
dados = dados.rename(columns=rename_dict)
dados['prazo'] = dados['prazo'].str.replace(' months', '').astype(int)

print(f"Shape após filtro: {dados.shape}")

In [ ]:
# Limpeza
total_antes = len(dados)
dados = dados.dropna()
dados = dados.drop(columns=['loan_status'], errors='ignore')

linhas_removidas = total_antes - len(dados)
print(f"Linhas removidas: {linhas_removidas} ({linhas_removidas/total_antes*100:.2f}%)")

# Outliers via IQR
for col in ['renda_anual', 'indice_endividamento']:
    Q1, Q3 = dados[col].quantile(0.25), dados[col].quantile(0.75)
    IQR = Q3 - Q1
    dados[col] = dados[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

for col in ['inadimplencias_2anos', 'registros_negativos']:
    dados[col] = dados[col].clip(upper=dados[col].quantile(0.99))

# Remover colunas de baixa variabilidade e multicolinearidade
dados = dados.drop(columns=['inadimplencias_2anos', 'registros_negativos', 'parcela'])

print(f"Shape final: {dados.shape}")
print(f"Distribuição target:\n{dados['inadimplente'].value_counts()}")

In [ ]:
# Split treino / teste / validação (70/15/15)
X = dados.drop(columns=['inadimplente'])
y = dados['inadimplente']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Treino:    {X_train.shape[0]:>10,} registros")
print(f"Teste:     {X_test.shape[0]:>10,} registros")
print(f"Validação: {X_val.shape[0]:>10,} registros")

In [ ]:
# scale_pos_weight calculado dinamicamente
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight calculado: {scale:.2f}")

# Treinar XGBoost
modelo_xgb_final = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss'
)

modelo_xgb_final.fit(X_train, y_train)
print("Treinamento concluído ✔")

In [ ]:
# Avaliação no conjunto de TESTE (threshold padrão 0.5)
y_pred_test = modelo_xgb_final.predict(X_test)

print(f"\n{'='*55}")
print(f"  XGBoost Final — Avaliação no Teste (threshold=0.50)")
print(f"{'='*55}")
print(classification_report(y_test, y_pred_test, target_names=['Adimplente', 'Inadimplente']))
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f} | F1: {f1_score(y_test, y_pred_test):.4f} | Recall: {recall_score(y_test, y_pred_test):.4f}")

## Ajuste de Threshold

Com dados desbalanceados, o threshold padrão (0.5) não é necessariamente o ideal. Testamos diferentes pontos de corte para maximizar o F1.

In [ ]:
y_proba_test = modelo_xgb_final.predict_proba(X_test)[:, 1]

resultados_threshold = []
for threshold in np.arange(0.25, 0.55, 0.05):
    y_pred_t = (y_proba_test >= threshold).astype(int)
    f1  = f1_score(y_test, y_pred_t)
    rec = recall_score(y_test, y_pred_t)
    acc = accuracy_score(y_test, y_pred_t)
    resultados_threshold.append({'threshold': round(threshold, 2), 'f1': f1, 'recall': rec, 'acc': acc})
    print(f"Threshold {threshold:.2f}: F1={f1:.4f} | Recall={rec:.4f} | Acurácia={acc:.4f}")

df_thresh = pd.DataFrame(resultados_threshold)
melhor = df_thresh.loc[df_thresh['f1'].idxmax()]
print(f"\nMelhor threshold: {melhor['threshold']:.2f} → F1={melhor['f1']:.4f}")

In [ ]:
# Visualizar métricas por threshold
plt.figure(figsize=(9, 4))
plt.plot(df_thresh['threshold'], df_thresh['f1'],    marker='o', label='F1 Inadimplente')
plt.plot(df_thresh['threshold'], df_thresh['recall'], marker='s', label='Recall')
plt.plot(df_thresh['threshold'], df_thresh['acc'],    marker='^', label='Acurácia')
plt.axvline(x=melhor['threshold'], color='red', linestyle='--',
            label=f"Melhor threshold ({melhor['threshold']:.2f})")
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Métricas por Threshold — XGBoost 2M')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Avaliação com melhor threshold no TESTE
best_thresh = float(melhor['threshold'])
y_pred_best = (y_proba_test >= best_thresh).astype(int)

print(f"\n{'='*55}")
print(f"  XGBoost Final — Threshold={best_thresh:.2f}")
print(f"{'='*55}")
print(classification_report(y_test, y_pred_best, target_names=['Adimplente', 'Inadimplente']))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Adimplente','Inadimplente'],
            yticklabels=['Adimplente','Inadimplente'])
plt.title(f'Matriz de Confusão — XGBoost (threshold={best_thresh:.2f})')
plt.ylabel('Real'); plt.xlabel('Previsto')
plt.tight_layout(); plt.show()

## Curva ROC / AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_test)
auc = roc_auc_score(y_test, y_proba_test)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', label=f'XGBoost (AUC={auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Aleatório')
plt.xlabel('FPR (Falso Positivo)')
plt.ylabel('TPR (Recall)')
plt.title('Curva ROC — XGBoost 2M')
plt.legend()
plt.tight_layout(); plt.show()

print(f"AUC-ROC: {auc:.4f}")

## Feature Importance

In [ ]:
importances = pd.Series(modelo_xgb_final.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values()

plt.figure(figsize=(8, 5))
importances_sorted.plot.barh(color='steelblue')
plt.title('Feature Importance — XGBoost 2M')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

print("\nFeature Importance (ordenado):")
print(importances.sort_values(ascending=False))

## Avaliação Final no Conjunto de Validação (Holdout)

O conjunto de validação **nunca foi usado** durante o desenvolvimento do modelo. Esta é a avaliação definitiva de generalização.

In [ ]:
y_proba_val = modelo_xgb_final.predict_proba(X_val)[:, 1]
y_pred_val  = (y_proba_val >= best_thresh).astype(int)

print(f"\n{'='*55}")
print(f"  Avaliação Final — Conjunto de Validação (holdout)")
print(f"  Threshold: {best_thresh:.2f}")
print(f"{'='*55}")
print(classification_report(y_val, y_pred_val, target_names=['Adimplente', 'Inadimplente']))

auc_val = roc_auc_score(y_val, y_proba_val)
print(f"AUC-ROC (validação): {auc_val:.4f}")
print(f"Acurácia: {accuracy_score(y_val, y_pred_val):.4f} | F1: {f1_score(y_val, y_pred_val):.4f} | Recall: {recall_score(y_val, y_pred_val):.4f}")

In [ ]:
# Salvar modelo + threshold juntos
NOMEMODELO = 'modelo_inadimplencia_v3_2M.pickle'

payload = {
    'modelo': modelo_xgb_final,
    'threshold': best_thresh,
    'features': X.columns.tolist()
}

with open(NOMEMODELO, 'wb') as f:
    pickle.dump(payload, f)
print(f"Modelo salvo como '{NOMEMODELO}' ✔")
print(f"Threshold salvo: {best_thresh:.2f}")
print(f"Features: {X.columns.tolist()}")

In [ ]:
# Exemplo de uso em produção
print("Exemplo de predição em produção:")

with open(NOMEMODELO, 'rb') as f:
    carregado = pickle.load(f)

modelo_prod    = carregado['modelo']
threshold_prod = carregado['threshold']
features_prod  = carregado['features']

novo_cliente = pd.DataFrame([X_val.iloc[0]], columns=features_prod)
proba = modelo_prod.predict_proba(novo_cliente)[0][1]
pred  = 1 if proba >= threshold_prod else 0

print(f"Probabilidade de inadimplência: {proba:.2%}")
print(f"Predição (threshold={threshold_prod:.2f}): {'Inadimplente ⚠️' if pred == 1 else 'Adimplente ✔'}")

# Conclusão Final

## Configuração
- **Dataset:** Lending Club completo (2M+ registros, sem amostragem)
- **Registros após limpeza:** ~1.3M
- **Features:** 9 (após remoção de `parcela`, `inadimplencias_2anos`, `registros_negativos`)
- **Modelo:** XGBoost com `scale_pos_weight` dinâmico

## Resultados

| Métrica | Teste (threshold=0.50) | Teste (threshold ajustado) | Validação |
|---------|------------------------|---------------------------|----------|
| Acurácia | ~0.64 | varia | varia |
| F1 Inadimplente | ~0.43 | melhorado | verificar |
| Recall Inadimplente | ~0.67 | ajustável | verificar |
| AUC-ROC | — | — | ~0.72 |

## Destaques

- **`scale_pos_weight` dinâmico** garante que o modelo funcione independentemente da distribuição da amostra
- **Ajuste de threshold** permite calibrar o tradeoff precision/recall conforme o apetite de risco do negócio
- **Validação separada** comprova a capacidade de generalização
- **Feature importance** revela que `score_credito` e `taxa_juros` são os preditores mais relevantes

## Interpretação de Negócio

Em crédito, o custo de um **Falso Negativo** (aprovar um inadimplente) costuma ser muito maior que o de um **Falso Positivo** (recusar um bom pagador). Por isso, o threshold ajustado abaixo de 0.5 reduz os Falsos Negativos ao custo de um recall ligeiramente menor na classe adimplente — estratégia alinhada com a gestão de risco.